<a href="https://colab.research.google.com/github/Apur52027/Machine-learing/blob/main/Week_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset
url = "https://raw.githubusercontent.com/Apur52027/Dataset_ML/main/diabetes.csv"

columns = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"
]

df = pd.read_csv(url, names=columns, skiprows=1)

print(df.head())
df.shape

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  


(768, 9)

In [5]:
x =df.drop(columns='Outcome')
y=df['Outcome']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [7]:
scaling = StandardScaler()
X_train = scaling.fit_transform(X_train)
X_test = scaling.transform(X_test)

### Numpy to Tensor Conversion

In [22]:
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()

In [14]:
y_train_tensor = torch.from_numpy(y_train.values).float().view(-1,1)
y_test_tensor = torch.from_numpy(y_test.values).float().view(-1,1)

In [15]:
y_test_tensor

tensor([[0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
      

In [17]:
class OurNN() :

  def __init__(self,input_size):
    self.weights = torch.rand(input_size,1,requires_grad = True)
    self.bias = torch.zeros(1,requires_grad=True)
  def forward(self,x):
    y_pred = torch.matmul(x,self.weights) + self.bias
    return torch.sigmoid(y_pred)
  def loss(self,y_pred,y):
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred,epsilon,1-epsilon)
    loss = -(y*torch.log(y_pred) + (1-y)*torch.log(1-y_pred))
    return loss.mean()




In [29]:
# hyperparameters
learning_rate = 0.01
epochs = 50

In [30]:
size = X_train_tensor.shape[1]
size

8

In [31]:
# creating model instance
model = OurNN(X_train_tensor.shape[1])

In [32]:
for epoch in range(epochs):
  # 1.forward pass
  y_pred = model.forward(X_train_tensor)
  # 2.loss
  loss =model.loss(y_pred,y_train_tensor)
  # 3.backward pass
  loss.backward()
  # 4.update
  with torch.no_grad():
    model.weights -= learning_rate * model.weights.grad
    model.bias -= learning_rate * model.bias.grad
  # 5. zero gradient
  model.weights.grad.zero_()
  model.bias.grad.zero_()
  print(f'epoch : {epoch}, loss : {loss.item()}')

epoch : 0, loss : 0.6597170233726501
epoch : 1, loss : 0.6591647863388062
epoch : 2, loss : 0.6586134433746338
epoch : 3, loss : 0.6580633521080017
epoch : 4, loss : 0.6575145721435547
epoch : 5, loss : 0.6569672226905823
epoch : 6, loss : 0.6564210057258606
epoch : 7, loss : 0.6558761596679688
epoch : 8, loss : 0.6553325057029724
epoch : 9, loss : 0.6547901034355164
epoch : 10, loss : 0.654248833656311
epoch : 11, loss : 0.6537090539932251
epoch : 12, loss : 0.6531704664230347
epoch : 13, loss : 0.652633011341095
epoch : 14, loss : 0.6520969271659851
epoch : 15, loss : 0.6515619158744812
epoch : 16, loss : 0.6510283946990967
epoch : 17, loss : 0.6504958868026733
epoch : 18, loss : 0.6499648094177246
epoch : 19, loss : 0.6494350433349609
epoch : 20, loss : 0.6489064693450928
epoch : 21, loss : 0.648378849029541
epoch : 22, loss : 0.6478525996208191
epoch : 23, loss : 0.6473277807235718
epoch : 24, loss : 0.6468040347099304
epoch : 25, loss : 0.6462815403938293
epoch : 26, loss : 0.6457

In [33]:
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred_class = (y_pred>0.5 ).float()
    accuracy = (y_pred_class == y_test_tensor).float().mean()

    print(f"Accuracy : {accuracy.item()}")

Accuracy : 0.701298713684082
